In [1]:
import os
import sys
import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download, HfApi

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print("Working directory:", os.getcwd())
print("Python:", sys.version)

Working directory: /home/samin/Desktop/mlops/MLOps-Course/HW03/HW3_Student/HW3_A
Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]


# Step 1: Download model from HuggingFace

Download `sentence-transformers/all-MiniLM-L6-v2` and save 6 files to `bundle/model/`:

| # | File |
|---|------|
| 1 | `config.json` |
| 2 | `tokenizer_config.json` |
| 3 | `tokenizer.json` |
| 4 | `vocab.txt` |
| 5 | `special_tokens_map.json` |
| 6 | `model.safetensors` |

Use `snapshot_download` from `huggingface_hub` or download manually with `AutoModel.save_pretrained()` + `AutoTokenizer.save_pretrained()`.

After downloading, save the git commit hash to `bundle/model/.commit`.

In [2]:
# Step 1: download 6 model files to bundle/model/

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
    "model.safetensors",
]

snapshot_download(
    repo_id=MODEL_ID,
    revision="main",
    local_dir=str(BUNDLE_MODEL_DIR),
    allow_patterns=REQUIRED_FILES,
)
print(f"Downloaded to {BUNDLE_MODEL_DIR.resolve()}")

# save HuggingFace commit hash
commit = HfApi().model_info(MODEL_ID, revision="main").sha
(BUNDLE_MODEL_DIR / ".commit").write_text(commit)
print(f"Commit hash: {commit}")

# verify all required files
REQUIRED = ["config.json", "tokenizer_config.json", "tokenizer.json",
            "vocab.txt", "special_tokens_map.json", "model.safetensors"]
for fname in REQUIRED:
    fpath = BUNDLE_MODEL_DIR / fname
    assert fpath.exists(), f"MISSING: {fname}"
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  [OK] {fname:35s} {size_mb:7.1f} MB")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Downloaded to /home/samin/Desktop/mlops/MLOps-Course/HW03/HW3_Student/HW3_A/bundle/model
Commit hash: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
  [OK] config.json                             0.0 MB
  [OK] tokenizer_config.json                   0.0 MB
  [OK] tokenizer.json                          0.4 MB
  [OK] vocab.txt                               0.2 MB
  [OK] special_tokens_map.json                 0.0 MB
  [OK] model.safetensors                      86.7 MB


# Step 2: Write metadata.json

Create `bundle/metadata.json` with accurate information about the bundle.
Required fields:
- `model_name`: the HuggingFace model ID
- `model_revision`: the git commit hash from the `.commit` file
- `embedding_dim`: 384
- `max_seq_len`: 256
- `framework_version`: the installed torch version
- `transformers_version`: the installed transformers version
- `built_by`: YOUR NAME
- `build_timestamp_utc`: current time in ISO 8601 UTC format

In [3]:
# Step 2: write bundle/metadata.json

import importlib.metadata

commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

metadata = {
    "model_name": MODEL_ID,
    "model_revision": commit,
    "embedding_dim": 384,
    "max_seq_len": 256,
    "framework": "pytorch",
    "framework_version": torch.__version__,
    "transformers_version": importlib.metadata.version("transformers"),
    "pooling": "mean-weight-by-attention-mask",
    "normalization": "l2",
    "built_by": "samin kakaei",
    "build_timestamp_utc": ts,
    "notes": "Mean-pool then L2-normalize. See predict.py for the 7-step pipeline.",
}

# write file
meta_path = Path("bundle/metadata.json")
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "model_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "embedding_dim": 384,
  "max_seq_len": 256,
  "framework": "pytorch",
  "framework_version": "2.7.1+cpu",
  "transformers_version": "4.54.1",
  "pooling": "mean-weight-by-attention-mask",
  "normalization": "l2",
  "built_by": "samin kakaei",
  "build_timestamp_utc": "2026-06-21T21:09:43Z",
  "notes": "Mean-pool then L2-normalize. See predict.py for the 7-step pipeline."
}


# Step 3: Write MANIFEST.json

Compute SHA-256 hash for every file under `bundle/` (except MANIFEST.json itself)
and write the manifest to `bundle/MANIFEST.json`.

The format must be:
```json
{
  "format_version": 1,
  "files": {
    "relative/path/to/file": "sha256hexdigest",
    ...
  }
}
```

Pro tip: you can peek at `scripts/gen_manifest.py` for reference.

In [6]:
# Step 3: SHA-256 hash every file under bundle/

def sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

root = Path("bundle")
files = {}
for p in sorted(root.rglob("*")):
    if p.is_file() and p.name != "MANIFEST.json":
        rel = str(p.relative_to(root))
        files[rel] = sha256(p)

manifest = {
    "format_version": 1,
    "files": files,
}

# write MANIFEST.json
manifest_path = Path("bundle/MANIFEST.json")
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"MANIFEST.json written with {len(manifest['files'])} files")
for rel, h in sorted(manifest["files"].items()):
    print(f"  {h[:16]}...  {rel}")

MANIFEST.json written with 25 files
  adf7b67e0f0083d8...  __pycache__/predict.cpython-310.pyc
  167bacd2350833cd...  metadata.json
  684888c0ebb17f37...  model/.cache/huggingface/.gitignore
  e3b0c44298fc1c14...  model/.cache/huggingface/download/config.json.lock
  6a674d636e1f6369...  model/.cache/huggingface/download/config.json.metadata
  e3b0c44298fc1c14...  model/.cache/huggingface/download/model.safetensors.lock
  fe0a25b6f84489dd...  model/.cache/huggingface/download/model.safetensors.metadata
  e3b0c44298fc1c14...  model/.cache/huggingface/download/special_tokens_map.json.lock
  15148030c49e8acb...  model/.cache/huggingface/download/special_tokens_map.json.metadata
  e3b0c44298fc1c14...  model/.cache/huggingface/download/tokenizer.json.lock
  566c85ff6eebb36f...  model/.cache/huggingface/download/tokenizer.json.metadata
  e3b0c44298fc1c14...  model/.cache/huggingface/download/tokenizer_config.json.lock
  8da2e0a17b435233...  model/.cache/huggingface/download/tokenizer_config.j

# Step 4: Write predict.py

Write `bundle/predict.py` with exactly 4 functions:

| Function | Signature | Returns |
|----------|-----------|---------|
| `load_bundle()` | no args | `(model, tokenizer)` tuple |
| `embed(texts)` | `List[str]` | `np.ndarray` shape `(N, 384)` float32 |
| `similarity(a, b)` | two `np.ndarray` | `float` (cosine similarity) |
| `info()` | no args | `dict` with metadata |

The **7-step pipeline** inside `embed()`:
1. Tokenize: `tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")`
2. Move tensors to device (cpu or cuda)
3. Forward pass under `torch.no_grad()` → `last_hidden_state`
4. Mean-pool: `sum(H * mask) / sum(mask).clamp(min=1e-9)`
5. L2 normalize: `F.normalize(pooled, p=2, dim=1)`
6. Detach, move to CPU, convert to `np.float32`
7. Return ndarray

**Important rules:**
- DO NOT import `sentence-transformers`. Use raw `transformers` only.
- Set `torch.manual_seed(0)` for determinism.
- Call `model.eval()` before inference.
- A template already exists at `bundle/predict.py` — implement the 4 functions there, then run the next cell.

In [5]:
# Step 4: smoke-test bundle/predict.py

import sys
sys.path.insert(0, "bundle")

from predict import load_bundle, embed, similarity, info

model, tokenizer = load_bundle()
print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tokenizer).__name__)
print(info())

sample = embed(["I love this so much"])
print("Sample embedding shape:", sample.shape)
print("Sample similarity:", similarity(sample[0], sample[0]))

Model loaded: BertModel
Tokenizer loaded: BertTokenizerFast
{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim': 384, 'max_seq_len': 256, 'device': 'cpu', 'framework': 'pytorch', 'deterministic': True, 'bundle_dir': '/home/samin/Desktop/mlops/MLOps-Course/HW03/HW3_Student/HW3_A/bundle/model'}
Sample embedding shape: (1, 384)
Sample similarity: 1.0000001192092896


/home/samin/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


# Step 5: Run tests

Run all 4 test files. All tests must pass with green dots.

Tests check:
- **test_parity.py** (7 tests): Correct embedding shape, L2 normalization, similarity
- **test_tokenization.py** (5 tests): Tokenizer behavior, special tokens, round-trip
- **test_determinism.py** (1 test): Same input → same output every time
- **test_adversarial.py** (10 tests): Edge cases (unicode, numerics, long text, empty strings)

In [7]:
# Step 5: run all tests (expect 23 passed)
!cd ~/Desktop/mlops/MLOps-Course/HW03/HW3_Student/HW3_A && PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-8.3.3, pluggy-1.6.0 -- /bin/python
cachedir: .pytest_cache
rootdir: /home/samin/Desktop/mlops/MLOps-Course/HW03/HW3_Student/HW3_A
plugins: anyio-4.12.1, opik-1.8.20
collected 23 items                                                             

tests/test_adversarial.py::test_missing_bundle_dir PASSED                [  4%]
tests/test_adversarial.py::test_very_long_text PASSED                    [  8%]
tests/test_adversarial.py::test_unicode_text PASSED                      [ 13%]
tests/test_adversarial.py::test_numeric_text PASSED                      [ 17%]
tests/test_adversarial.py::test_single_token_text PASSED                 [ 21%]
tests/test_adversarial.py::test_duplicate_texts PASSED                   [ 26%]
tests/test_adversarial.py::test_batch_of_one PASSED                      [ 30%]
tests/test_adversarial.py::test_large_batch PASSED                      

# Step 6: Register in MLflow

Log the bundle to MLflow using `mlflow.sentence_transformers.log_model()`.

Before running this cell, make sure:
1. You have `source .env` (or set the env vars manually)
2. All tests pass
3. `bundle/model/` contains the 6 model files

In [ ]:
# Step 6: register bundle in MLflow (run once only)

import mlflow

# load credentials from .env
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))

with mlflow.start_run(run_name="hw3a-bundle") as run:
    mlflow.log_param("embedding_dim", 384)
    mlflow.log_param("max_seq_len", 256)
    mlflow.log_param("model_id", MODEL_ID)
    mlflow.set_tag("stage", "candidate")

    # log SentenceTransformer artifact (MLflow API expects loaded model)
    from sentence_transformers import SentenceTransformer

    st_model = SentenceTransformer(str(BUNDLE_MODEL_DIR.resolve()))
    model_info = mlflow.sentence_transformers.log_model(
        st_model,
        artifact_path="bundle",
        task="llm/v1/embeddings",
    )
    print("Model URI:", model_info.model_uri)
    print("Run ID:", run.info.run_id)
    print("Experiment:", os.environ.get("MLFLOW_EXPERIMENT_NAME"))
    print("MLflow UI:", os.environ["MLFLOW_TRACKING_URI"])

/home/samin/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
2026/06/22 00:32:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.7.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/22 00:33:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpi55tttio/model, flavor: sentence_transformers). Fall back to return ['sentence-transformers==5.6.0', 'transformers==4.54.1', 'torch==2.7.1']. Set logging level to DEBUG to see the full traceback. 


## Step 6 verify — pull model from MLflow (evidence cell)

Run this cell **instead of re-running Step 6** if registration already succeeded.
It fetches your latest finished `hw3a-bundle` run, lists artifacts, loads the model from MLflow, and prints proof output for mentors/judges.

In [8]:
# Step 6 verify: pull registered model from MLflow (evidence cell)

import mlflow
from mlflow.tracking import MlflowClient

# load credentials from .env
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())

experiment_name = os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder")
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

client = MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise RuntimeError(f"Experiment not found: {experiment_name}")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'hw3a-bundle' AND attributes.status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=1,
)
if not runs:
    raise RuntimeError("No FINISHED hw3a-bundle run found. Run Step 6 once, or check MLflow UI.")

run = runs[0]
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/bundle"

print("=== MLflow registration evidence ===")
print("Tracking URI:", os.environ["MLFLOW_TRACKING_URI"])
print("Experiment:", experiment_name)
print("Run ID:", run_id)
print("Run name:", run.data.tags.get("mlflow.runName"))
print("Status:", run.info.status)
print("Model URI:", model_uri)
print("UI:", f"{os.environ['MLFLOW_TRACKING_URI']}/#/experiments/{experiment.experiment_id}/runs/{run_id}")
print("\nParams:")
for key, value in sorted(run.data.params.items()):
    print(f"  {key} = {value}")
print("\nTags:")
for key, value in sorted(run.data.tags.items()):
    if not key.startswith("mlflow."):
        print(f"  {key} = {value}")

print("\nArtifacts under bundle/:")
for artifact in client.list_artifacts(run_id, path="bundle"):
    size = f"{artifact.file_size} bytes" if artifact.file_size else "dir"
    print(f"  {artifact.path} ({size})")

print("\nLoading model from MLflow...")
loaded_model = mlflow.sentence_transformers.load_model(model_uri)
sample = loaded_model.encode(["I love this so much"])
print("Loaded model type:", type(loaded_model).__name__)
print("Sample embedding shape:", sample.shape)
print("Sample embedding dtype:", sample.dtype)
print("Sample L2 norm:", float((sample[0] ** 2).sum() ** 0.5))
print("\nOK — model pulled successfully from MLflow.")

=== MLflow registration evidence ===
Tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw03_encoder_samin_kakaei
Run ID: 4f6a4c44605d468084c9a7bfe3e2e9aa
Run name: hw3a-bundle
Status: FINISHED
Model URI: runs:/4f6a4c44605d468084c9a7bfe3e2e9aa/bundle
UI: http://185.50.38.163:33014/#/experiments/61/runs/4f6a4c44605d468084c9a7bfe3e2e9aa

Params:
  embedding_dim = 384
  max_seq_len = 256
  model_id = sentence-transformers/all-MiniLM-L6-v2

Tags:
  stage = candidate

Artifacts under bundle/:
  bundle/MLmodel (891 bytes)
  bundle/conda.yaml (187 bytes)
  bundle/model.sentence_transformer (dir)
  bundle/python_env.yaml (115 bytes)
  bundle/requirements.txt (77 bytes)

Loading model from MLflow...


2026/06/22 00:44:19 INFO mlflow.sentence_transformers: 'runs:/4f6a4c44605d468084c9a7bfe3e2e9aa/bundle' resolved as 'mlflow-artifacts:/61/4f6a4c44605d468084c9a7bfe3e2e9aa/artifacts/bundle'


Loaded model type: SentenceTransformer
Sample embedding shape: (1, 384)
Sample embedding dtype: float32
Sample L2 norm: 4.115846361189992

OK — model pulled successfully from MLflow.


/home/samin/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


# Step 7: Upload to MinIO

Upload your `bundle/` directory to the shared MinIO bucket.

Run the provided upload script:
```bash
source .env && bash scripts/01_upload_to_minio.sh
```

Or from this notebook:

In [9]:
# Step 7: upload bundle to MinIO
!set -a && source .env && set +a && bash scripts/01_upload_to_minio.sh

Configuring MinIO alias 'qbc12'...
]11;?\Bucket: hw03-bundles
Uploading bundle/ → s3://hw03-bundles/samin_kakaei/
...ements.txt: 87.34 MiB / 87.34 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 937.31 KiB/s 1m35s
=== Verification ===
Files on MinIO:
[2026-06-22 00:46:17 +0330] 2.8KiB STANDARD MANIFEST.json
[2026-06-22 00:46:18 +0330] 4.0KiB STANDARD __pycache__/predict.cpython-310.pyc
[2026-06-22 00:46:18 +0330]   502B STANDARD metadata.json
[2026-06-22 00:46:18 +0330]     1B STANDARD model/.cache/huggingface/.gitignore
[2026-06-22 00:46:18 +0330]     0B STANDARD model/.cache/huggingface/download/config.json.lock
[2026-06-22 00:46:18 +0330]   101B STANDARD model/.cache/huggingface/download/config.json.metadata
[2026-06-22 00:46:18 +0330]     0B STANDARD model/.cache/huggingface/download/model.safetensors.lock
[2026-06-22 00:46:18 +0330]   125B STANDARD model/.cache/huggingface/download/model.safetensors.metadata
[2026-06-22 00:46:18 +0330]     0B STANDARD model/.cache/huggingface/download/special_toke

## Submission Checklist

Before submitting, verify:

- [ ] `encoder_bundle.ipynb` — all cells executed with visible outputs
- [ ] `bundle/predict.py` — 4 functions implemented
- [ ] `bundle/metadata.json` — all fields filled (no TODO placeholders)
- [ ] `bundle/MANIFEST.json` — real SHA-256 hashes for every file
- [ ] `bundle/requirements.txt` — pinned dependencies listed
- [ ] `bundle/model/` — 6 model files present
- [ ] `EVIDENCE/pytest_pass.png` — all tests green
- [ ] `EVIDENCE/mlflow_registered.png` — MLflow UI showing the model
- [ ] `EVIDENCE/minio_upload.png` — MinIO upload confirmation

Good luck!